In [ ]:
# 匯入模組：json 用於解析 JSON 檔案，pprint 用於格式化輸出
import json
from pprint import pprint

# 讀取空氣品質 JSON 檔案，取得所有測站的空氣品質監測資料
with open("空氣品質aqi.json",encoding="utf-8") as file:
    data:dict = json.load(file)

# 從 JSON 資料中取出測站記錄列表，每個元素為一個 dict
contents:list[dict]= data['records']

# 以格式化方式印出所有測站資料，方便檢視內容
pprint(contents)


### Exampe Json file
```
{'aqi': '64',
  'co': '0.18',
  'co_8hr': '0.1',
  'county': '屏東縣',
  'datacreationdate': '2024-03-15 09:00',
  'latitude': '22.260899',
  'longitude': '120.651472',
  'no': '0.3',
  'no2': '2.1',
  'nox': '2.4',
  'o3': '58.1',
  'o3_8hr': '60.0',
  'pm10': '25',
  'pm10_avg': '25',
  'pm2.5': '14',
  'pm2.5_avg': '12.5',
  'pollutant': '臭氧八小時',
  'siteid': '313',
  'sitename': '屏東(枋山)',
  'so2': '0.4',
  'so2_avg': '0',
  'status': '普通',
  'unit': '',
  'winddirec': '91',
  'windspeed': '7.3'}
```

In [8]:
# 匯入 Pydantic 相關工具與 datetime 模組
from pydantic import BaseModel,Field,field_validator
from datetime import datetime

# 定義空氣品質測站的資料模型，對應 JSON 中每個測站的欄位
class AirSite(BaseModel):
    aqi:int                                       # 空氣品質指標 AQI
    county:str                                    # 縣市名稱
    date:datetime = Field(alias="datacreationdate")  # 資料建立時間，對應原始欄位 datacreationdate
    lat:float = Field(alias = "latitude")         # 緯度
    lon:float = Field(alias="longitude")          # 經度
    pm25:float = Field(alias="pm2.5")             # PM2.5 濃度
    pollutant:str                                 # 主要污染物名稱
    site_name:str = Field(alias="sitename")       # 測站名稱
    status:str                                    # 空氣品質狀態（良好/普通/對敏感族群不健康等）

    # 自定義驗證器：將空字串轉換為 0，避免型別轉換錯誤
    @field_validator('aqi','lat','lon','pm25', mode='before')
    @classmethod
    def empty_to_zero(cls, v):
        return 0 if v == '' else v

In [9]:
# 定義 Root 模型：作為整個 JSON 回應的根結構
class Root(BaseModel):
    status:bool = True            # 回應狀態，預設為 True 表示成功
    sites:list[AirSite]           # 存放所有測站的列表

In [10]:
# 將 JSON 資料逐一轉換為 AirSite 物件，再組裝成 Root 模型
# **item 會將 dict 的每個 key-value 作為參數傳入 AirSite()
root = Root(sites=[AirSite(**item) for item in contents])


In [ ]:
for site in root.sites:
    print(site)